# FeTA — Finding a balanced data split

We have 80 FeTA subjects. We want a **60/20/20 train/validation/test split**, with similar ages, pathology groups and reconstruction methods in each set.

This notebook is just for checking the split. The final code is in `src/splits.py` and `scripts/create_splits.py`.

## 1. Look at the gestational ages

We group gestational ages into four ranges. We also check pathology, since younger and older subjects may have different pathology distributions.

In [1]:
import pandas as pd
from pathlib import Path

# Load FeTA metadata.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

metadata_path = PROJECT_ROOT / "data" / "FeTA" / "feta_2.4" / "participants.tsv"
df = pd.read_csv(metadata_path, sep="\t")

# Group gestational ages into 4 intervals.
age_bins = [20, 24, 28, 32, 36]
age_labels = ["20-24", "24-28", "28-32", "32-36"]

df["age_group"] = pd.cut(
    df["gestational_age"],
    bins=age_bins,
    labels=age_labels,
    right=False,
)

print("Subjects per age group:")
print(df["age_group"].value_counts().sort_index())

print("\nAge group vs pathology:")
print(pd.crosstab(df["age_group"], df["pathology"]))

Subjects per age group:
age_group
20-24    20
24-28    31
28-32    20
32-36     9
Name: count, dtype: int64

Age group vs pathology:
pathology  Neurotypical  Pathological
age_group                            
20-24                 4            16
24-28                 9            22
28-32                11             9
32-36                 7             2


**What we found**
- All 80 subjects have gestational age metadata.
- The groups contain **20, 31, 20 and 9** subjects.
- Pathology is not evenly spread across ages, so we should check both variables.

## 2. First try: stratify by age

We first use `seed=42`. This keeps the age groups well represented, but pathology is still a bit uneven between train and test.

In [2]:
from sklearn.model_selection import train_test_split

# First split: 60% train, 40% remaining.
train_df, remaining_df = train_test_split(
    df,
    test_size=0.4,
    stratify=df["age_group"],
    random_state=42,
)

# Second split: 20% validation, 20% test.
val_df, test_df = train_test_split(
    remaining_df,
    test_size=0.5,
    stratify=remaining_df["age_group"],
    random_state=42,
)

for name, subset in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df),
]:
    print(f"\n{name}: {len(subset)} subjects")
    print(subset["age_group"].value_counts().sort_index())
    print(subset["pathology"].value_counts())


Train: 48 subjects
age_group
20-24    12
24-28    19
28-32    12
32-36     5
Name: count, dtype: int64
pathology
Pathological    32
Neurotypical    16
Name: count, dtype: int64

Validation: 16 subjects
age_group
20-24    4
24-28    6
28-32    4
32-36    2
Name: count, dtype: int64
pathology
Pathological    9
Neurotypical    7
Name: count, dtype: int64

Test: 16 subjects
age_group
20-24    4
24-28    6
28-32    4
32-36    2
Name: count, dtype: int64
pathology
Neurotypical    8
Pathological    8
Name: count, dtype: int64


## 3. Check the reconstruction methods

FeTA uses two reconstruction methods: MIAL and IRTK. We read them from the MRI filenames rather than guessing from subject numbers.

In [3]:
# Read the real reconstruction method from the preprocessed MRI filenames.
feta_dir = PROJECT_ROOT / "data" / "processed" / "feta_fomo"
image_paths = sorted(feta_dir.glob("sub-*/**/anat/*_T2w.nii.gz"))

reconstruction_by_subject = {}

for path in image_paths:
    subject = next(part for part in path.parts if part.startswith("sub-"))
    filename = path.name.lower()

    if "rec-mial" in filename:
        method = "MIAL"
    elif "rec-irtk" in filename:
        method = "IRTK"
    else:
        raise ValueError(f"Unknown reconstruction method: {path}")

    if subject in reconstruction_by_subject:
        raise ValueError(f"Duplicate subject: {subject}")

    reconstruction_by_subject[subject] = method

assert set(reconstruction_by_subject) == set(df["participant_id"])
print("Reconstruction methods:", pd.Series(reconstruction_by_subject).value_counts().to_dict())

def get_reconstruction(participant_id):
    return reconstruction_by_subject[participant_id]

Reconstruction methods: {'MIAL': 40, 'IRTK': 40}


## 4. Find a better split

We try different random seeds, always stratifying by age. We first look for a good pathology balance, then a good MIAL/IRTK balance.

We only use metadata to choose the seed — **not model results**.

In [4]:
# Search for a split balanced in age, pathology and reconstruction.
target_pathology = [29, 10, 10]
target_mial = [24, 8, 8]

best_score = float("inf")
best_seed = None
best_split = None

for seed in range(2000):
    train, remaining = train_test_split(
        df, test_size=0.4,
        stratify=df["age_group"],
        random_state=seed,
    )

    val, test = train_test_split(
        remaining, test_size=0.5,
        stratify=remaining["age_group"],
        random_state=seed,
    )

    splits = [train, val, test]

    pathology_counts = [
        (s["pathology"] == "Pathological").sum()
        for s in splits
    ]

    mial_counts = [
        (s["participant_id"].apply(get_reconstruction) == "MIAL").sum()
        for s in splits
    ]

    # Prioritize pathology balance, then reconstruction balance.
    pathology_error = sum(
        abs(a - b)
        for a, b in zip(pathology_counts, target_pathology)
    )
    reconstruction_error = sum(
        abs(a - b)
        for a, b in zip(mial_counts, target_mial)
    )

    score = (pathology_error, reconstruction_error)

    if best_split is None or score < best_score:
        best_score = score
        best_seed = seed
        best_split = {
            "Train": train,
            "Validation": val,
            "Test": test,
        }

    if score == (0, 0):
        break

print("Best seed:", best_seed)
print("Pathology error, reconstruction error:", best_score)

for name, subset in best_split.items():
    print(f"\n{name}: {len(subset)} subjects")
    print("Age groups:", subset["age_group"].value_counts().sort_index().to_dict())
    print("Pathology:", subset["pathology"].value_counts().to_dict())
    print("Reconstruction:",
          subset["participant_id"].apply(get_reconstruction).value_counts().to_dict())
    print("Mean GA:", round(subset["gestational_age"].mean(), 2))

Best seed: 367
Pathology error, reconstruction error: (np.int64(0), np.int64(0))

Train: 48 subjects
Age groups: {'20-24': 12, '24-28': 19, '28-32': 12, '32-36': 5}
Pathology: {'Pathological': 29, 'Neurotypical': 19}
Reconstruction: {'MIAL': 24, 'IRTK': 24}
Mean GA: 26.89

Validation: 16 subjects
Age groups: {'20-24': 4, '24-28': 6, '28-32': 4, '32-36': 2}
Pathology: {'Pathological': 10, 'Neurotypical': 6}
Reconstruction: {'MIAL': 8, 'IRTK': 8}
Mean GA: 27.01

Test: 16 subjects
Age groups: {'20-24': 4, '24-28': 6, '28-32': 4, '32-36': 2}
Pathology: {'Pathological': 10, 'Neurotypical': 6}
Reconstruction: {'MIAL': 8, 'IRTK': 8}
Mean GA: 27.49


**Best split: `seed=367`**

| | Train | Validation | Test |
|---|---:|---:|---:|
| Subjects | 48 | 16 | 16 |
| Pathological | 29 | 10 | 10 |
| Neurotypical | 19 | 6 | 6 |
| MIAL | 24 | 8 | 8 |
| IRTK | 24 | 8 | 8 |

All four age groups are represented in each split. This looks more balanced than our first try.

## 5. Check the exact ages

Age groups are balanced, but we also want to check the real minimum, maximum, mean and median ages.

In [5]:
# Check gestational age coverage in each split.
for name, subset in best_split.items():
    ages = subset["gestational_age"]

    print(f"\n{name}")
    print("Minimum GA:", ages.min())
    print("Maximum GA:", ages.max())
    print("Mean GA:", round(ages.mean(), 2))
    print("Median GA:", round(ages.median(), 2))


Train
Minimum GA: 20.1
Maximum GA: 34.8
Mean GA: 26.89
Median GA: 27.0

Validation
Minimum GA: 21.8
Maximum GA: 32.8
Mean GA: 27.01
Median GA: 26.75

Test
Minimum GA: 20.0
Maximum GA: 34.8
Mean GA: 27.49
Median GA: 27.8


**Age check:** train and test cover nearly the full gestational age range. The mean ages are also close (about 27 weeks). The validation range is a little narrower.

## 6. Compare with the saved JSON

We check that the IDs saved by `scripts/create_splits.py` are exactly the same as in this notebook. This avoids silently changing the split.

In [6]:

import json

# Load the final FeTA splits generated by our script.
split_path = PROJECT_ROOT / "data" / "splits" / "feta_splits.json"

with open(split_path, encoding="utf-8") as f:
    saved_splits = json.load(f)

# Compare the saved splits with our notebook.
for notebook_name, json_name in [
    ("Train", "train"),
    ("Validation", "val"),
    ("Test", "test"),
]:
    notebook_ids = set(best_split[notebook_name]["participant_id"])
    saved_ids = set(saved_splits[json_name])

    assert notebook_ids == saved_ids, (
        f"Different subjects in {json_name}!"
    )

    print(f"{json_name}: {len(saved_ids)} subjects — identical ✅")

print("\nAll FeTA splits match the notebook!")


train: 48 subjects — identical ✅
val: 16 subjects — identical ✅
test: 16 subjects — identical ✅

All FeTA splits match the notebook!


## Conclusion

We keep **60/20/20 with `seed=367`** for FeTA. The age groups, pathology and reconstruction methods are well balanced, and the saved JSON matches the notebook.

The split is created at **subject level**, with no subject overlap. We can now use `data/splits/feta_splits.json` for the next steps.